# SpecklePy Packfiles for Analytics

This notebook shows how to:

1. Authenticate with SpecklePy.
2. Resolve IDs from environment input.
3. Check dataset availability.
4. Download main and EAV datasets.
5. Query them with DuckDB.

## Scope notes

- Current example queries are Revit-oriented (`category`, `proxy.level`).
- Older published versions (from before automatic dataset generation) can fail with availability/download errors.

## Setup with `.env`

Create a `.env` file next to this notebook:

```bash
SPECKLE_TOKEN=your_personal_access_token
SPECKLE_MODEL_URL=https://app.speckle.systems/projects/your_project_id/models/your_model_id
# Optional:
# SPECKLE_HOST=https://app.speckle.systems
# SPECKLE_VERSION_ID=optional_specific_version
```

If you need to create a token first, see [Building with PATs](https://docs.speckle.systems/developers/authentication/pats).

Notebook flow assumes token and model URL come from environment variables.

In [ ]:
%pip install -q specklepy duckdb pandas requests python-dotenv

In [ ]:
import os
from pathlib import Path
from urllib.parse import urlparse

import duckdb
import pandas as pd
import requests
from dotenv import load_dotenv
from specklepy.api.client import SpeckleClient

load_dotenv()

In [ ]:
HOST = os.getenv("SPECKLE_HOST", "https://app.speckle.systems")
TOKEN = os.getenv("SPECKLE_TOKEN")
MODEL_URL = os.getenv("SPECKLE_MODEL_URL", "").strip()
VERSION_ID = os.getenv("SPECKLE_VERSION_ID", "").strip() or None

if not TOKEN:
    raise ValueError("Set SPECKLE_TOKEN in your .env file.")
if not MODEL_URL:
    raise ValueError("Set SPECKLE_MODEL_URL in your .env file.")

parsed = urlparse(MODEL_URL)
parts = [p for p in parsed.path.split("/") if p]
if len(parts) < 4 or parts[0] != "projects" or parts[2] != "models":
    raise ValueError("SPECKLE_MODEL_URL must look like /projects/{projectId}/models/{modelRef}")

PROJECT_ID = parts[1]
model_ref = parts[3]
if "@" in model_ref:
    MODEL_ID, parsed_version_id = model_ref.split("@", 1)
    VERSION_ID = VERSION_ID or parsed_version_id
else:
    MODEL_ID = model_ref

client = SpeckleClient(host=HOST)
client.authenticate_with_token(TOKEN)
print(f"Authenticated. project={PROJECT_ID} model={MODEL_ID} version={VERSION_ID or 'latest'}")

In [ ]:
def graphql_post(query: str, variables: dict) -> dict:
    response = requests.post(
        f"{HOST}/graphql",
        headers={"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"},
        json={"query": query, "variables": variables},
        timeout=60,
    )
    response.raise_for_status()
    payload = response.json()
    if payload.get("errors"):
        raise RuntimeError(payload["errors"])
    return payload["data"]

if not VERSION_ID:
    latest_query = """
    query LatestVersion($projectId: String!, $modelId: String!) {
      project(id: $projectId) {
        model(id: $modelId) {
          versions(limit: 1) {
            items {
              id
            }
          }
        }
      }
    }
    """
    latest_data = graphql_post(latest_query, {"projectId": PROJECT_ID, "modelId": MODEL_ID})
    items = latest_data["project"]["model"]["versions"]["items"]
    if not items:
        raise RuntimeError("No versions found for this model.")
    VERSION_ID = items[0]["id"]

availability_query = """
query PackfileAvailability($projectId: String!, $modelId: String!, $versionId: String!) {
  project(id: $projectId) {
    model(id: $modelId) {
      version(id: $versionId) {
        id
        objectKey
        packfileSize
        referencedObject
        createdAt
      }
    }
  }
}
"""

availability_data = graphql_post(
    availability_query,
    {"projectId": PROJECT_ID, "modelId": MODEL_ID, "versionId": VERSION_ID},
)
version = availability_data["project"]["model"]["version"]
if not version:
    raise RuntimeError("Version not found.")
if not version.get("objectKey"):
    raise RuntimeError(
        "Primary dataset not available for this version. "
        "This often means the version is historical (published before auto-generation)."
    )

print(f"Using version: {VERSION_ID}")
version

In [ ]:
output_dir = Path("packfiles") / PROJECT_ID / MODEL_ID / VERSION_ID
output_dir.mkdir(parents=True, exist_ok=True)

main_url = f"{HOST}/api/v1/projects/{PROJECT_ID}/models/{MODEL_ID}/versions/{VERSION_ID}/download"
eav_url = f"{HOST}/api/v1/projects/{PROJECT_ID}/models/{MODEL_ID}/versions/{VERSION_ID}/eav/download"

main_path = output_dir / f"{VERSION_ID}.duckdb"
eav_path = output_dir / f"{VERSION_ID}.eav.duckdb"
headers = {"Authorization": f"Bearer {TOKEN}"}

def download_file(url: str, target: Path) -> None:
    with requests.get(url, headers=headers, stream=True, timeout=300) as response:
        response.raise_for_status()
        with target.open("wb") as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)

download_file(main_url, main_path)

try:
    download_file(eav_url, eav_path)
    has_eav = True
except requests.HTTPError as error:
    has_eav = False
    print(f"EAV download unavailable: {error}")

main_path, (eav_path if has_eav else None)

In [ ]:
con = duckdb.connect()
con.execute(f"ATTACH '{main_path.as_posix()}' AS main_pf (READ_ONLY)")

if has_eav:
    con.execute(f"ATTACH '{eav_path.as_posix()}' AS eav_pf (READ_ONLY)")
    summary = con.execute(
        """
        SELECT
          (SELECT COUNT(*) FROM main_pf.objects) AS object_count,
          (SELECT COUNT(*) FROM eav_pf.properties) AS property_rows,
          (SELECT COUNT(*) FROM eav_pf.proxies) AS proxy_rows
        """
    ).fetchdf()
else:
    summary = con.execute(
        """
        SELECT
          (SELECT COUNT(*) FROM main_pf.objects) AS object_count,
          NULL::BIGINT AS property_rows,
          NULL::BIGINT AS proxy_rows
        """
    ).fetchdf()

summary

In [ ]:
if not has_eav:
    raise RuntimeError("EAV artefact not attached. Category query needs eav_pf.properties.")

category_counts = con.execute(
    """
    SELECT value_text AS category, COUNT(*) AS count
    FROM eav_pf.properties
    WHERE path = 'category' AND value_text IS NOT NULL
    GROUP BY value_text
    ORDER BY count DESC
    LIMIT 25
    """
).fetchdf()

category_counts

In [ ]:
if not has_eav:
    raise RuntimeError("EAV artefact not attached. Level query needs eav_pf.properties.")

level_counts = con.execute(
    """
    SELECT value_text AS level_name, COUNT(*) AS count
    FROM eav_pf.properties
    WHERE path = 'proxy.level' AND value_text IS NOT NULL
    GROUP BY value_text
    ORDER BY count DESC
    LIMIT 25
    """
).fetchdf()

level_counts